In [44]:
import torch
import torch.nn as nn # neural network
import torch.nn.functional as F # activation functions
from torch.utils.data import DataLoader, TensorDataset # data loader

from sklearn.datasets import load_breast_cancer # The dataset itself
from sklearn.model_selection import train_test_split # to split the dataset into training and testing sets
from sklearn.preprocessing import StandardScaler # to standardize the dataset


In [ ]:
# x, y are numpy arrays, y is [0/1] for malignant(cancerous)/benign(non-cancerous)
x, y = load_breast_cancer(return_X_y=True) # load the dataset

# split the data into training and testing sets -> 20% for testing and the rest for training
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)


scaler = StandardScaler() # create a scaler object for standardization (put all measurements on a similar scale e.g., 0-1)


x_train_scaled = scaler.fit_transform(x_train)  # Learn the average and standard deviation of each measurement from the training data, then use them to standardize the training data.

x_test_scaled = scaler.transform(x_test)  # Standardize the testing data using the same averages and standard deviations learned from the training data, without calculating new ones.

In [48]:
# Transfer it to tensors
x_train_scaled_tensor = torch.from_numpy(x_train_scaled).float()  
x_test_scaled_tensor = torch.from_numpy(x_test_scaled).float()  

# Add an extra dimension to the tensor to make it compatible with the model's output shape
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)  
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)  

In [49]:
# Create a dataset object for the training data
train_dataset = TensorDataset(x_train_scaled_tensor, y_train_tensor) 

In [50]:
x_train_scaled_tensor.shape, y_train_tensor.shape, x_test_scaled_tensor.shape, y_test_tensor.shape

(torch.Size([455, 30]),
 torch.Size([455, 1]),
 torch.Size([114, 30]),
 torch.Size([114, 1]))

In [51]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  
# Create a data loader for the training dataset with a batch size of 32 and shuffling enabled

In [52]:
# Create our neural network 
class BCNet(nn.Module): # Breast Cancer Net
    def __init__(self):
        super(BCNet, self).__init__()
        # fully connected, 30 -> we have 30 features/instance 
        #                  64 -> 64 neurons in the first hidden layer (output of it to the next one )
        self.fc1 = nn.Linear(30, 64) 
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        
    def forward(self, x):
        x = self.fc1(x) # pass the input through the first layer
        # We have to break the linearity of the model, otherwise it will just be a linear regression model.
        # via applying a non-linear activation function to the output of the first layer
        x = F.relu(x) # ReLU activation function
        
        # Now apply to all other layers
        x = F.relu(self.fc2(x))
        x = F.sigmoid(self.fc3(x)) # the values between 0-1, which is what we want for a binary classification problem
        
        return x

In [54]:
model = BCNet() # create an instance of the model

In [55]:
# create our criterion -> binary cross entropy loss function, which is suitable for binary classification problems
criterion = nn.BCELoss() # Binary Cross Entropy Loss 

# Create the optimizer 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Adam optimizer with a learning rate of 0.001

In [56]:
# Make the training
epochs = 20 # epochs = number of times the model will see the entire dataset

for epoch in range(epochs): # number of epochs
    model.train() # set the model to training mode
    running_loss = 0.0 # initialize the running loss for this epoch
    
    for x_batch, y_batch in train_loader: # iterate over the batches of data
        optimizer.zero_grad() # zero the gradients for this batch
        
        predication = model(x_batch) # forward pass: compute the model output for this batch
        loss = criterion(predication, y_batch) # compute the loss for this batch
        
        loss.backward() # backward pass: compute the gradients (differentiation)
        optimizer.step() # update the model parameters
        
        running_loss += loss.item() * x_batch.size(0) # accumulate the loss for this epoch
        
    # print the average loss for this epoch
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader.dataset):.4f}") 

Epoch 1/20, Loss: 0.6308
Epoch 2/20, Loss: 0.4907
Epoch 3/20, Loss: 0.3208
Epoch 4/20, Loss: 0.1961
Epoch 5/20, Loss: 0.1376
Epoch 6/20, Loss: 0.1073
Epoch 7/20, Loss: 0.0887
Epoch 8/20, Loss: 0.0774
Epoch 9/20, Loss: 0.0702
Epoch 10/20, Loss: 0.0625
Epoch 11/20, Loss: 0.0579
Epoch 12/20, Loss: 0.0533
Epoch 13/20, Loss: 0.0497
Epoch 14/20, Loss: 0.0469
Epoch 15/20, Loss: 0.0442
Epoch 16/20, Loss: 0.0412
Epoch 17/20, Loss: 0.0388
Epoch 18/20, Loss: 0.0363
Epoch 19/20, Loss: 0.0340
Epoch 20/20, Loss: 0.0322


In [59]:
# Evaluate the neural network on the test set
with torch.no_grad(): # we don't need to compute gradients for evaluation
    model.eval() # set the model to evaluation mode
    
    predictions = model(x_test_scaled_tensor) # get the model predictions for the test set
    loss = criterion(predictions, y_test_tensor).item() # compute the loss for the test set
    print(f"Test Loss: {loss:.4f}") # print the loss
    
    # compute the accuracy for the test set
    accuracy  = ((predictions >= 0.5) == y_test_tensor).float().mean().item()  
    print(f"Test Accuracy: {accuracy:.4f}") # print the accuracy

Test Loss: 0.0626
Test Accuracy: 0.9825
